In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import pyplot as plt
from pathlib import Path
from tqdm import tqdm
import itertools
import dask
from dask.distributed import Client
import dask.array as da
from numba import njit

from biosnicar.drivers import get_albedo, setup_snicar
import warnings

In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
@njit
def albedo_bc_grid(bc, sit):
    if not np.isnan(bc) and not np.isnan(sit):
        impurities = impurities_def
        impurities[0].conc[0]=bc
        ice = ice_def 
        ice.dz[0]=sit
        albedo = get_albedo.get_grid("adding-doubling", ice,
                                 illumination,rt_config,model_config,plot_config,
                                 impurities, plot=False,
                                 validate=True)
        bba=albedo.bba
    else:
        bba=np.nan
    return bba

In [32]:
(ice_def, illumination, rt_config, model_config, plot_config, impurities_def) = setup_snicar.setup_snicar('./biosnicar/inputs_orig.yaml')

In [2]:
df = xr.open_dataset('df_bio_1.nc') # bc_conc, sisnthick. bc_conc calculated in snow AND 10cm of sea ice 
#df = df.sel(time='2022-02-01')
#df['bba'] = 0 * df['BC_conc']
df = df.to_dataarray()
df

<xarray.DataArray (variable: 2, time: 168, lat: 26, lon: 360)> Size: 25MB
array([[[[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         ...,
         [0.29765483, 0.30032283, 0.30044424, ..., 0.29779539,
          0.30008625, 0.29894979],
         [0.27691098, 0.27462038, 0.27336696, ..., 0.2823656 ,
          0.27962004, 0.27907272],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]],

        [[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
...
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]],

        [[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         ...,
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]]]])
Coordinates:
  * time      (time) datetime64[ns] 1kB 2010-01-01T00:30:00 ... 2023-12-01T00...
  * lat       (lat) int64 208B 65 66 67 68 69 70 71 72 ... 84 85 86 87 88 89 90
  * lon       (lon) int64 3kB -180 -179 -178 -177 -176 ... 175 176 177 178 179
  * variable  (variable) object 16B 'BC_conc' 'sisnthick'
Attributes: (12/33)
    CDI:                               Climate Data Interface version 1.9.8 (...
    Conventions:                       CF-1
    History:                           Wed May  8 12:25:17 2024: ncrcat HTTP_...
    Filename:                          MERRA2_100.tavgM_2d_adg_Nx.198001.nc4
    Comment:                           GMAO filename: d5124_m2_jan79.tavg1_2d...
    Institution:                       NASA Global Modeling and Assimilation ...
    ...                                ...
    identifier_product_doi:            10.5067/RZIK2TV7PP38
    RangeBeginningTime:                00:00:00.000000
    RangeEndingTime:                   23:59:59.000000
    history_L34RS:                     'Created by L34RS v1.4.3 @ NASA GES DI...
    CDO:                               Climate Data Operators version 1.9.8 (...
    NCO:                               netCDF Operators version 5.0.7 (Homepa...

In [3]:
# access dimensions of dataframe
lengths = np.shape(df) # var=2, time=168 (14 years, 12 months), 26 latitudes, 360 longitudes

n_time = lengths[1]
n_lat = lengths[2]
n_lon = lengths[3]

# dimensions dataset: var (bc=0, sn=1), time (i), lat (j), lon (k)
total_it = n_time * n_lat * n_lon

In [15]:
@njit
def modif_input(bc,sit):
    file1 = open("./biosnicar-py/biosnicar/inputs.yaml", "w") 
    lst = []
    k = 0
    with open('./biosnicar-py/biosnicar/inputs_orig.yaml', 'r') as fh:
        for line in fh:
            if line[:5]=='  BC:':
                k = 1
            if (line[:10]=='    CONC: ') and (k==1):
                k = 0
                lst.append('    CONC: ['+str(bc)+']\n')
            elif line[:5]=='  DZ:':
                lst.append('  DZ: ['+str(sit)+']\n')
            else:
                lst.append(line)
    file1.writelines(lst) 
    file1.close() 

@njit
def albedo_bc():
    albedo = get_albedo.get("adding-doubling", plot=False, validate=True)
    return albedo.BBA

In [16]:
# initialize matrices 
out_alb = np.zeros((n_time, n_lat, n_lon))
out_alb_nobc = np.zeros((n_time, n_lat, n_lon))
bc0 = np.zeros((n_time, n_lat, n_lon))

with tqdm(total = total_it, desc="Processing", leave=True) as pbar:
    # Iterate over all combinations of the last three dimensions
    for i, j, k in itertools.product(range(n_time), range(n_lat), range(n_lon)):
        if not (np.isnan(df.data[0,i, j, k]) | np.isnan(df.data[1,i, j, k])):
            modif_input(df.data[0,i, j, k], df.data[1,i, j, k])
            out_alb[i,j,k] = albedo_bc()
            modif_input(bc0[i,j,k], df.data[1,i, j, k])
            out_alb_nobc[i,j,k] = albedo_bc()
        pbar.update(1)

Processing:   0%|          | 371/1572480 [00:00<00:16, 95942.21it/s]


UnsupportedError: Failed in nopython mode pipeline (step: analyzing bytecode)
[1mThe 'with (context manager) as (variable):' construct is not supported.[0m

In [41]:
# initialize matrices 
out_alb = np.zeros((n_time, n_lat, n_lon))
out_alb_nobc = np.zeros((n_time, n_lat, n_lon))
bc0 = np.zeros((n_time, n_lat, n_lon))

with tqdm(total = total_it, desc="Processing", leave=True) as pbar:
    # Iterate over all combinations of the last three dimensions
    for i, j, k in itertools.product(range(n_time), range(n_lat), range(n_lon)):
        out_alb[i,j,k] = albedo_bc_grid(df.data[0,i, j, k], df.data[1,i, j, k])
        out_alb_nobc[i,j,k] = albedo_bd_grid(bc0[i, j, k], df.data[1,i, j, k])
        pbar.update(1)

Processing:   0%|          | 0/1572480 [00:00<?, ?it/s]


TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1mUntyped global name 'impurities_def':[0m [1mCannot type list element type <class 'biosnicar.classes.impurity.Impurity'>
[1m
File "../../../../../../tmp/ipykernel_17811/999760966.py", line 4:[0m
[1m<source missing, REPL/exec in use?>[0m
[0m

In [34]:
xr.apply_ufunc(

<xarray.DataArray (variable: 2, time: 168, lat: 26, lon: 360)> Size: 25MB
array([[[[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         ...,
         [0.29765483, 0.30032283, 0.30044424, ..., 0.29779539,
          0.30008625, 0.29894979],
         [0.27691098, 0.27462038, 0.27336696, ..., 0.2823656 ,
          0.27962004, 0.27907272],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]],

        [[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
...
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]],

        [[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         ...,
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]]]])
Coordinates:
  * time      (time) datetime64[ns] 1kB 2010-01-01T00:30:00 ... 2023-12-01T00...
  * lat       (lat) int64 208B 65 66 67 68 69 70 71 72 ... 84 85 86 87 88 89 90
  * lon       (lon) int64 3kB -180 -179 -178 -177 -176 ... 175 176 177 178 179
  * variable  (variable) object 16B 'BC_conc' 'sisnthick'
Attributes: (12/33)
    CDI:                               Climate Data Interface version 1.9.8 (...
    Conventions:                       CF-1
    History:                           Wed May  8 12:25:17 2024: ncrcat HTTP_...
    Filename:                          MERRA2_100.tavgM_2d_adg_Nx.198001.nc4
    Comment:                           GMAO filename: d5124_m2_jan79.tavg1_2d...
    Institution:                       NASA Global Modeling and Assimilation ...
    ...                                ...
    identifier_product_doi:            10.5067/RZIK2TV7PP38
    RangeBeginningTime:                00:00:00.000000
    RangeEndingTime:                   23:59:59.000000
    history_L34RS:                     'Created by L34RS v1.4.3 @ NASA GES DI...
    CDO:                               Climate Data Operators version 1.9.8 (...
    NCO:                               netCDF Operators version 5.0.7 (Homepa...

In [61]:
np.nanmin(out_alb-out_alb_nobc)

-0.01655126369843607

In [66]:
# Create Dask arrays for your output matrices
out_alb = da.zeros((n_time, n_lat, n_lon), chunks=(1, n_lat, n_lon))
out_alb_nobc = da.zeros((n_time, n_lat, n_lon), chunks=(1, n_lat, n_lon))
bc0 = da.zeros((n_time, n_lat, n_lon), chunks=(1, n_lat, n_lon))

# A function to compute the outputs based on input data
def compute_albedo(i, j, k):
    if not (np.isnan(float(df[0, i, j, k])) or np.isnan(float(df[1, i, j, k]))):
        modif_input(float(df[0, i, j, k]), float(df[1, i, j, k]))
        alb_bc_value = albedo_bc()
        modif_input(bc0[i, j, k], float(df[1, i, j, k]))
        alb_nobc_value = albedo_bc()
        return (alb_bc_value, alb_nobc_value)
    else:
        return (np.nan, np.nan)

tasks = []
for i, j, k in itertools.product(range(n_time), range(n_lat), range(n_lon)):
    tasks.append(dask.delayed(compute_albedo)(i, j, k))

# Execute the tasks in parallel
results = dask.compute(*tasks)

# Now, store the results into the corresponding output arrays
for index, (alb_bc_value, alb_nobc_value) in enumerate(results):
    i, j, k = np.unravel_index(index, (n_time, n_lat, n_lon))
    out_alb[i, j, k] = alb_bc_value
    out_alb_nobc[i, j, k] = alb_nobc_value

# Compute the final arrays
out_alb = out_alb.compute()
out_alb_nobc = out_alb_nobc.compute()

TypeError: 'NoneType' object is not subscriptable

In [47]:
sum=0
for i, j, k in itertools.product(range(n_time), range(n_lat), range(n_lon)):
    if sum==3045:
        print(i,j,k)
    sum += 1

0 8 165


In [12]:
client = Client()

# Define your own code
def f(i, j, k):
    out_alb = 0
    out_alb_nobc = 0
    if not (np.isnan(float(df[0,i, j, k])) | np.isnan(float(df[1,i, j, k]))):
        modif_input(float(df[0,i,j,k]), float(df[1,i,j,k]))
        out_alb[i,j,k] = albedo_bc()
        modif_input(bc0[i,j,k], float(df[1,i,j,k]))
        out_alb_nobc[i,j,k] = albedo_bc()
    return (out_alb, out_alb_nobc)

out_alb = np.zeros((n_time, n_lat, n_lon))
out_alb_nobc = np.zeros((n_time, n_lat, n_lon))
bc0 = np.zeros((n_time, n_lat, n_lon))

# Run your code in parallel
#futures = client.map(f, range(n_time), range(n_lat), range(n_lon))

futures = []
for i in range(n_time):
    for j in range(n_lat):
        for k in range(n_lon):
            futures.append(client.submit(f, i, j, k))

results = client.gather(futures)

for i, j, k, out_value, out_nobc_value in results:
    out_alb[i, j, k] = out_value
    out_alb_nobc[i, j, k] = out_nobc_value
    
client.close()

/opt/conda/envs/albedomodel/lib/python3.11/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46013 instead
  warnings.warn(
2025-05-13 11:08:05,026 - distributed.worker - WARNING - Compute Failed
Key:       f-57c29bf0d234ae614ad3c8de8c956011
Function:  f
args:      (0, 2, 0)
kwargs:    {}
Exception: "IndexError('index 2 is out of bounds for axis 2 with size 2')"

2025-05-13 11:08:05,027 - distributed.worker - WARNING - Compute Failed
Key:       f-ccdfd91e629fbdee3df4f6dde74acd02
Function:  f
args:      (0, 4, 0)
kwargs:    {}
Exception: "IndexError('index 4 is out of bounds for axis 2 with size 2')"

2025-05-13 11:08:05,027 - distributed.worker - WARNING - Compute Failed
Key:       f-d196b1e8c8de1b434a3d34493b835da5
Function:  f
args:      (0, 6, 0)
kwargs:    {}
Exception: "IndexError('index 6 is out of bounds for axis 2 with size 2')"

2025-05-13 11:08:05,037 - distributed.worke

IndexError: index 2 is out of bounds for axis 2 with size 2

2025-05-13 11:08:05,123 - distributed.worker - WARNING - Compute Failed
Key:       f-cc4e56c566b4fedbd9875c5459d429ca
Function:  f
args:      (0, 3, 1)
kwargs:    {}
Exception: "IndexError('index 3 is out of bounds for axis 2 with size 2')"

2025-05-13 11:08:05,123 - distributed.worker - WARNING - Compute Failed
Key:       f-dc391d10592e52779686029973953e3f
Function:  f
args:      (0, 5, 1)
kwargs:    {}
Exception: "IndexError('index 5 is out of bounds for axis 2 with size 2')"

2025-05-13 11:08:05,127 - distributed.worker - WARNING - Compute Failed
Key:       f-1693e7368e961b9768bd4336409528ec
Function:  f
args:      (0, 7, 1)
kwargs:    {}
Exception: "IndexError('index 7 is out of bounds for axis 2 with size 2')"

2025-05-13 11:08:05,128 - distributed.worker - WARNING - Compute Failed
Key:       f-133f63bfcf729dedf60796c0d2a3a15c
Function:  f
args:      (0, 9, 1)
kwargs:    {}
Exception: "IndexError('index 9 is out of bounds for axis 2 with size 2')"

2025-05-13 11:08:05,128 - distri

In [33]:
sw = xr.open_dataset('../data_for_group3/MORE_DATA/SWDOWN_MONTHLY_ERA5.nc') # from ERA5 - W/m2
sw = sw.sel(valid_time=sw.valid_time.dt.season=='JJA').mean()
sw

<xarray.Dataset> Size: 16B
Dimensions:     ()
Coordinates:
    number      int64 8B ...
Data variables:
    avg_sdswrf  float64 8B 194.4

In [34]:
rf = (out_alb_nobc-out_alb)*sw.avg_sdswrf.values
rf

array([0.0018297 , 0.01827944, 0.18198773])